# Entretien technique Oplit — Atelier & temps de production

**Le contexte.** Oplit aide des usines à **ordonnancer leur production**. Un nouveau client (un atelier d'usinage / soudure / assemblage) veut qu'on l'aide à planifier ses opérations. On va partir de *son* problème, pas du code.

## Partie A — Ordonnancement de l'atelier (flow shop)

L'atelier fabrique 3 types de pièces (A, B, C) sur **2 machines**. Chaque pièce passe sur la Machine 1 **puis** la Machine 2 (dans cet ordre). Une machine ne traite qu'une pièce à la fois.

| Pièce | Machine 1 | Machine 2 |
|-------|-----------|-----------|
| A     | 2 h       | 3 h       |
| B     | 4 h       | 1 h       |
| C     | 3 h       | 2 h       |

Le responsable d'atelier dit : *« Je veux juste tout sortir le plus vite possible. »*

💬 **À discuter (pas de modèle mathématique attendu) :**
- Que cherche-t-on à optimiser, dans les mots du client puis dans les vôtres ?
- Qu'est-ce qu'on **décide** réellement ici ? Quelles **contraintes** ne peut-on pas violer ?
- Quelle **règle simple** (heuristique) un planificateur humain pourrait-il appliquer ? Pourquoi serait-elle raisonnable ?
- De façon générale : face à *n'importe quel* problème d'optimisation, quelle est votre **check-list** avant d'écrire la moindre ligne de code ?

## Partie B — Prédire les temps de traitement

Problème : tout ce qu'on vient de faire suppose qu'on **connaît** la durée de chaque opération (2 h, 4 h…). Dans le monde réel, la même opération prend un temps différent selon la pièce, la machine, la taille du lot, et même la situation de l'usine à l'instant t.

Le client n'a pas de temps standards propres — il a un **export brut de ce qui s'est réellement passé** sur l'atelier. Donc le problème d'ordonnancement dépend en silence d'un problème de **prédiction** : *combien de temps va durer cette opération ?*

Voici un échantillon de l'export client. **Réagissez à ce que vous voyez.**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Marche en local (repo cloné) ET sur Google Colab (téléchargement auto depuis GitHub)
LOCAL_PATH = "../data/operations_raw.csv"
RAW_URL = "https://raw.githubusercontent.com/alexandrerioo/scheduling_sandbox/main/data/operations_raw.csv"
CSV = LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL

df = pd.read_csv(CSV, dtype=str)
print(f"{len(df):,} lignes  x  {df.shape[1]} colonnes")
df.head(8)

### Glossaire des colonnes

| Colonne | Description |
|---|---|
| `of_id`, `step_no` | Ordre de Fabrication et n° d'étape dans la gamme |
| `article_ref`, `article_family` | Référence article et famille (Usinage, Soudure, Assemblage, Finition) |
| `quantity` | Quantité de l'opération |
| `machine_id`, `machine_type` | Machine physique et son type (vitesse différente) |
| `planned_start_ts`, `planned_end_ts`, `planned_duration_min` | Plan **prévu** par le planificateur (temps standard) |
| `actual_start_ts`, `actual_end_ts` | Début / fin **réels** de l'opération |
| **`actual_processing_time_min`** | **Durée réelle de l'opération — la cible à prédire** |
| `delay_min` | Retard = fin réelle − fin prévue |
| `status` | done / running / aborted |
| `scrap_qty` | Quantité rebutée pendant l'opération |
| `record_created_ts` | Horodatage d'écriture de la ligne par le MES |

In [4]:
# Volumétrie & période
# NB : les formats de date ne sont pas homogènes selon la source -> on parse les deux
iso = pd.to_datetime(df["planned_start_ts"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
fr  = pd.to_datetime(df["planned_start_ts"], format="%d/%m/%Y %H:%M", errors="coerce")
ps = iso.fillna(fr)
print("Période :", ps.min(), "->", ps.max())
print("OF distincts        :", df["of_id"].nunique())
print("Références article   :", df["article_ref"].nunique(), "(formes de surface)")
print("Machines / types     :", df["machine_id"].nunique(), "/", df["machine_type"].nunique())
print("Cible manquante      :", df["actual_processing_time_min"].isna().sum(), "lignes")
print("Lignes dupliquées    :", df.duplicated().sum())

Période : 2024-01-01 06:39:00 -> 2024-09-01 12:35:30
OF distincts        : 6000
Références article   : 92 (formes de surface)
Machines / types     : 12 / 4
Cible manquante      : 847 lignes
Lignes dupliquées    : 228


### 💬 À discuter avec votre interlocuteur

Pas besoin de tout coder : on veut votre **démarche** et vos **arbitrages**.

1. **Ingestion & qualité.** Le vrai fichier client fait des dizaines de Go et ne tient pas en mémoire. Comment l'ingérez-vous ? Et ces données sont incomplètes/incohérentes. *Que faites-vous en premier* ?
2. **Cadrage.** Quelle est exactement la cible ? À quel **grain** modélise-t-on ? Quelles colonnes **ne pouvez-vous pas** utiliser pour prédire, et pourquoi ?
3. **Split.** Comment séparez-vous train / test ? (deux ans d'historique, on veut prédire le futur)
4. **Features & modèle.** Quelles variables construisez-vous ? Quel premier modèle, et quelle **référence (baseline)** à battre ?
5. **Évaluation.** Comment savez-vous que le modèle est bon, et comment l'expliquez-vous au responsable d'atelier ?
6. **Production.** Une fois déployé, le modèle nourrit l'ordonnanceur de la Partie A. Que peut-il se passer en prod, et qu'y gagne le client ?

In [ ]:
# Zone de travail libre
